# AI Agent + 외부 API 도구 실습 (DuckDuckGo / Open-Meteo)

**Tool Calling Agent**의 Mock 도구들을 모두 **실제 외부 API 도구**로 교체합니다.
모두 **API 키 발급이 불필요한 무료 공개 API**만 사용합니다.

---
**주요 실습 내용**
- **DuckDuckGo 웹 검색** — `duckduckgo-search` (실시간 검색)
- **Open-Meteo 날씨 API** — 좌표 검색(geocoding) + 현재 날씨(forecast), HTTP GET 만으로 호출
- 위 2개 외부 도구를 한 Agent 안에 묶고, 질문 유형별로 자동 선택되는지 확인

---
**4번 노트북과의 차이**

| 항목 | 4번 노트북 | 5번 노트북 |
|------|-----------|-----------|
| 날씨 | Mock 딕셔너리 (서울/부산/제주/대전) | **Open-Meteo** 실시간 (전 세계 도시) |
| 웹 검색 | 없음 | **DuckDuckGo** 실시간 검색 |

## 1. 패키지 설치 및 환경 설정

- `duckduckgo-search` — DuckDuckGo 검색
- Open-Meteo는 별도 SDK 없이 `requests` 로 HTTP 호출만 하면 되므로 추가 설치가 필요 없습니다.

In [ ]:
%pip install -r ../requirements.txt
%pip install -q duckduckgo-search 
%pip install -U ddgs

from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
assert openai_api_key, "OPENAI_API_KEY 환경변수가 필요합니다."

from langchain_openai import ChatOpenAI
import langchain

llm = ChatOpenAI(model="gpt-4o", openai_api_key=openai_api_key, temperature=0)
print(f"LangChain 버전: {langchain.__version__}")
print("환경 설정 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 47.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.

## 2. DuckDuckGo 웹 검색 도구 단독 테스트

LangChain Community 패키지에는 DuckDuckGo 검색을 감싼 래퍼가 두 종류 있습니다.

| 클래스 | 반환 형식 | 특징 |
|--------|----------|------|
| `DuckDuckGoSearchRun` | 검색 결과 본문을 합친 **문자열** | 가볍게 요약 답변용 |
| `DuckDuckGoSearchResults` | 제목/URL/본문이 포함된 **리스트 문자열** | 출처(링크)를 함께 보여주고 싶을 때 |

두 가지 모두 사용해보고, Agent용 도구는 출처를 함께 반환하는 후자를 기반으로 만들겠습니다.

In [6]:
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

search_run = DuckDuckGoSearchRun()
search_results = DuckDuckGoSearchResults(num_results=4)

# 1) 단순 문자열 결과
print("[DuckDuckGoSearchRun]")
print(search_run.invoke("DuckDuckGo ?")[:500], "...\n")

# 2) 제목/링크/본문이 포함된 결과
print("[DuckDuckGoSearchResults]")
print(search_results.invoke("DuckDuckGo 에 대해 설명해줘")[:700], "...")

[DuckDuckGoSearchRun]
DuckDuckGo ... DuckDuckGo is an American software company focused on online privacy whose flagship product is a search engine named DuckDuckGo. Founded by Gabriel Weinberg in 2008, its later products include browser extensions and a custom DuckDuckGo web browser. DuckDuckGo logo DuckDuckGo (officially Duck Duck Go, Inc.) is an American software company founded in 2008 by Gabriel Weinberg that develops privacy-focused internet products, including a... That's why millions of people choose DuckDuck ...

[DuckDuckGoSearchResults]
snippet: 17 Oct 2025 · DuckDuckGo는 Bing인데, 프록시를 거친 거죠. Brave Search는 독립적이고, Google을 프록시로 사용할 수 있는 옵션이 있어요. Brave Browser는 사용자들이 자신의 기기에서 ..., title: DuckDuckGo vs. Brave Search : r/degoogle - Reddit, link: https://www.reddit.com/r/degoogle/comments/1o8zsbj/duckduckgo_vs_brave_search/?tl=ko, snippet: 14 Jan 2026 · 개인 블로그라는 작은 공간도, 콘텐츠의 출처와 맥락에 따라 전혀 예상하지 못한 경로로 연결될 수 있다는 점을 다시 한 번 실감한 순간이다. #DuckDuckGo · #네이버블로거, title: 오늘 개인 블로그를 살펴보다가 처음 보는 

## 3. `@tool` 로 감싸기

LangChain의 검색 래퍼를 그대로 Agent에 넘길 수도 있지만, **`@tool` 데코레이터로 한 번 감싸면**
- 도구 이름과 docstring을 한국어로 명확히 줄 수 있고,
- 결과 가공(상위 N개만, 길이 제한 등)을 자유롭게 할 수 있습니다.

여기서는 결과가 너무 길어 토큰을 낭비하지 않도록 **최대 1500자**로 자릅니다.

In [7]:
from langchain_core.tools import tool

_ddg = DuckDuckGoSearchResults(num_results=5)


@tool
def web_search(query: str) -> str:
    """DuckDuckGo 웹 검색을 수행합니다.
    최신 뉴스, 실시간 정보, LLM이 학습하지 못한 최근 사건, 외부 사이트의 사실 확인이 필요할 때 사용하세요.
    반환값에는 검색 결과의 제목(title), 본문(snippet), 출처 링크(link)가 포함됩니다.
    """
    try:
        raw = _ddg.invoke(query)
    except Exception as e:
        return f"검색 오류: {e}"
    # 토큰 절약을 위해 길이 제한
    return raw[:1500] if isinstance(raw, str) else str(raw)[:1500]


# 단독 테스트
print(web_search.invoke("한국 인공지능 산업 동향")[:800], "...")

snippet: 18 hours ago - 글로벌 산업 자동화 솔루션 기업 한국훼스토(대표 연승훈)가 오는 6월 10일부터 12일까지 서울 코엑스에서 개최되는 제15회 ‘STK 2026(스마트테크 코리아 2026)’에 참가해 제조 산업의 AI 전환을 이끌 혁신 기술을 선보인다고 밝혔다. STK 2026은 2011년 최초 개최 이후 인공지능(AI), 로보틱스, 클라우드 등 첨단 기술을 중심으로 산업 전반의 디지털 혁신 흐름을 조망해온 국내 대표 기술 전시회로, 올해는 산업 간 연결성과 AI 기반 제조 혁신을 핵심 테마로 개최된다., title: 한국훼스토, ‘STK 2026’ 참가…제조 AX의 새로운 가능성 제시 < 애플리케이션 < 인공지능 < 기사본문 - 로봇신문, link: https://www.irobotnews.com/news/articleView.html?idxno=46579, snippet: November 25, 2025 - 그 결과 한국은 내수 중심 제조와 규제 위주의 기술 정책에 머문 일부 선진 공업국과 달리, 안보·산업·기술이 서로 맞물려 움직이는 지속적 혁신 시스템을 구축하며 반도체·조선·배터리·방위 분야에서 국제 경쟁력을 확보하게 되었다. 반도체 산업에서는 이제 공정과 설계 전 과정에 인공지능이 들어오면서, 어떻게 더 많이, 더 정확하게 만들 것인가에 대한 기준이 달라지고 있다., title: [기고] 한국의 인공지능 전환점 2 '가치선도 문명 국가로' < 오피니언 < 뉴스 < 기사본문 - 헬로디디, link: https://www.hellodd.com/news/articleView.htm ...


## 4. Agent 루프 재정의

`run_agent` 함수를 가져옵니다.
LLM이 `tool_calls` 를 반환하지 않을 때까지 도구 실행을 반복하는 단순한 루프입니다.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage, AIMessage


def run_agent(user_input: str, tools: list, system: str = None, chat_history: list = None, verbose: bool = True) -> str:
    """Tool Calling Agent 루프 (4번 노트북과 동일)."""
    llm_with_tools = llm.bind_tools(tools)
    tools_map = {t.name: t for t in tools}

    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    if chat_history:
        messages.extend(chat_history)
    messages.append(HumanMessage(content=user_input))

    step = 1
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break

        for tc in response.tool_calls:
            tool_name = tc["name"]
            tool_args = tc["args"]
            tool_result = tools_map[tool_name].invoke(tool_args)

            if verbose:
                preview = str(tool_result)
                if len(preview) > 200:
                    preview = preview[:200] + " ..."
                print(f"  [Step {step}] Tool: {tool_name}({tool_args})")
                print(f"           → {preview}")

            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tc["id"]))
            step += 1

    return response.content

## 5. 추가 외부 API 도구 정의 

이번에는 Mock 도구(`get_weather`)를 **실제 외부 API**로 교체합니다.

| 도구 | 외부 API | 키 필요 여부 | 비고 |
|------|---------|---------|------|
| `get_weather` | [Open-Meteo](https://open-meteo.com) (geocoding + forecast) | ❌ | 도시명 → 위경도 → 현재 날씨 |

In [ ]:
import requests
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings


# --- (1) Open-Meteo 무료 날씨 API (API 키 불필요) -----------------------------
# 1) 도시명 → 위경도 (geocoding-api.open-meteo.com)
# 2) 위경도 → 현재 날씨 (api.open-meteo.com)

_WMO_CODES = {
    0: "맑음",
    1: "대체로 맑음", 2: "부분적으로 흐림", 3: "흐림",
    45: "안개", 48: "착빙성 안개",
    51: "약한 이슬비", 53: "이슬비", 55: "강한 이슬비",
    61: "약한 비", 63: "비", 65: "강한 비",
    71: "약한 눈", 73: "눈", 75: "강한 눈",
    80: "소나기", 81: "강한 소나기", 82: "매우 강한 소나기",
    95: "뇌우", 96: "우박 동반 뇌우", 99: "강한 뇌우와 우박",
}


@tool
def get_weather(city: str) -> str:
    """Open-Meteo 무료 API로 주어진 도시의 실제 현재 날씨를 조회합니다 (API 키 불필요).
    한국 도시는 한글, 해외 도시는 영문명을 권장합니다. 예: '서울', 'Tokyo', 'New York'.
    """
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "ko"},
            timeout=10,
        ).json()
        results = geo.get("results") or []
        if not results:
            return f"'{city}' 의 좌표를 찾을 수 없습니다."
        loc = results[0]
        lat, lon = loc["latitude"], loc["longitude"]
        name = loc.get("name", city)
        country = loc.get("country", "")

        fc = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lon,
                "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
                "timezone": "auto",
            },
            timeout=10,
        ).json()
        c = fc.get("current", {})
        desc = _WMO_CODES.get(c.get("weather_code"), f"코드 {c.get('weather_code')}")
        return (
            f"{name}({country}) 현재 날씨: {desc}, "
            f"기온 {c.get('temperature_2m')}°C, "
            f"습도 {c.get('relative_humidity_2m')}%, "
            f"풍속 {c.get('wind_speed_10m')} m/s "
            f"(관측: {c.get('time')})"
        )
    except Exception as e:
        return f"날씨 조회 오류: {e}"



In [21]:
# 단독 테스트
print("[get_weather]     ", get_weather.invoke("seoul"))

[get_weather]      서울특별시(대한민국) 현재 날씨: 대체로 맑음, 기온 21.9°C, 습도 81%, 풍속 5.1 m/s (관측: 2026-05-28T10:45)


## 6. Multi-tool Agent: 2개 외부 도구 합치기

이제 Agent에 도구 2개를 모두 등록합니다.
- `web_search` — **DuckDuckGo** 실시간 웹 검색
- `get_weather` — **Open-Meteo** 실시간 날씨


Agent는 질문 의도에 따라 다음과 같이 선택해야 합니다.
- *“오늘 LangChain 0.3 릴리스 변경점”* → `web_search`
- *“오사카 지금 날씨”* → `get_weather`

In [25]:
all_tools = [get_weather, web_search]

SYSTEM = (
    "당신은 유능한 AI 비서입니다. 도구를 적절히 선택해서 사용자를 도와주세요.\n"
    "- 최신 뉴스/실시간 정보/외부 사이트 사실 확인은 web_search 사용\n"
    "- 날씨 질문은 get_weather (Open-Meteo 실시간 데이터) 사용\n"
    "- 답변은 한국어로 하고, web_search 결과를 사용했다면 본문 끝에 출처 링크를 명시하세요."
)

### 6-1. 실시간 정보가 필요한 질문 (web_search 가 호출되어야 함)

In [26]:
print("=" * 60)
print("[질문] 최근 LangChain 0.3 릴리스에서 바뀐 점 요약해줘.")
print()
ans = run_agent(
    "최근 LangChain 0.3 릴리스에서 바뀐 점을 한국어로 요약해줘. 출처 링크도 같이.",
    all_tools,
    system=SYSTEM,
)
print(f"\n최종 답변:\n{ans}")

[질문] 최근 LangChain 0.3 릴리스에서 바뀐 점 요약해줘.

  [Step 1] Tool: web_search({'query': 'LangChain 0.3 release notes'})
           → snippet: LangChain version langchain-core==0.3.86 was officially released on May 07, 2026. At first glance, the release notes look small, title: LangChain 0.3.86 Released: Security Fix for AI Engineer ...

최종 답변:
LangChain 0.3 릴리스에서는 보안 수정이 주요 변경 사항으로 포함되었습니다. 이 릴리스는 AI 엔지니어들이 주목해야 할 보안 문제를 해결하는 데 중점을 두고 있습니다. LangChain은 600개 이상의 통합을 지원하는 오픈 소스 플랫폼으로, 애플리케이션을 전체적으로 다시 작성하지 않고도 구성 요소를 교체할 수 있도록 설계되었습니다. [출처 링크](https://www.skakarh.com/blog/langchain-new-release-security-fix-every-ai-engineer-should-notice)


### 6-2. 날씨 질문 (날씨 API 호출되어야 함, 웹 검색은 부르면 안 됨)

In [29]:
print("=" * 60)
print("제주도 날씨는?")
print()
ans = run_agent("제주도 날씨는?", all_tools, system=SYSTEM)
print(f"\n최종 답변:\n{ans}")

제주도 날씨는?

  [Step 1] Tool: get_weather({'city': '제주도'})
           → '제주도' 의 좌표를 찾을 수 없습니다.
  [Step 2] Tool: get_weather({'city': '제주'})
           → '제주' 의 좌표를 찾을 수 없습니다.

최종 답변:
죄송합니다. 현재 제주도의 날씨 정보를 가져오는 데 문제가 발생하고 있습니다. 잠시 후 다시 시도해 주시거나 다른 질문이 있으시면 말씀해 주세요.


### 6-3. 복합 질문 (웹 검색 + 날씨 조합)

Agent가 **여러 외부 API 도구를 순차적으로 호출**해 답을 만들 수 있는지 확인합니다.
예: 최근 활동은 웹 검색에서, 그 사람이 머무는 도시의 날씨는 Open-Meteo에서.

In [30]:
print("=" * 60)
print("[질문] 최신 'AI 알고리즘 윤리' 관련 웹 기사 + 런던 현재 날씨")
print()
ans = run_agent(
    "최근 'AI 알고리즘 윤리'와 관련된 뉴스 한두 건을 웹에서 찾아 정리해줘. "
    "그리고 런던 현재 날씨도 알려줘.",
    all_tools,
    system=SYSTEM,
)
print(f"\n최종 답변:\n{ans}")

[질문] 최신 'AI 알고리즘 윤리' 관련 웹 기사 + 런던 현재 날씨

  [Step 1] Tool: web_search({'query': 'AI 알고리즘 윤리 최근 뉴스'})
           → snippet: March 8, 2026 - 해당 회의에서는 AI 무기 체계의 신뢰성 확보, 인간 통제력 유지, 법적·윤리적 책임 문제 등이 핵심 의제로 다뤄졌습니다. 미국도 REAIM을 계기로 '군사 분야 AI의 책임 있는 사용을 위한 정치적 선언'을 주도하며 다수 국가의 참여를 독려하고 있습니다. 보안 및 디지털 정책 분야의 한 전문가는 "전장의 데이터 ...
  [Step 2] Tool: get_weather({'city': 'London'})
           → 날씨 조회 오류: HTTPSConnectionPool(host='geocoding-api.open-meteo.com', port=443): Read timed out. (read timeout=10)

최종 답변:
최근 'AI 알고리즘 윤리'와 관련된 뉴스는 다음과 같습니다:

1. **"AI가 좌표 찍으면 인간이 쏜다"…현실화한 '알고리즘 전쟁'**  
   이 기사에서는 AI 무기 체계의 신뢰성 확보, 인간 통제력 유지, 법적·윤리적 책임 문제 등이 논의되었습니다. 특히, 군사적 목적의 AI 활용 시 알고리즘의 신뢰성을 검증하고 인간의 책임 범위를 명확히 하는 국제적 통제 장치의 필요성이 강조되었습니다. [출처](https://news.nate.com/view/20260309n05194)

2. **윤리와 알고리즘, 기술이 정의를 침범할 때**  
   국내에서는 서울대 AI정책이니셔티브와 KAIST 휴머니즘 AI센터가 "설명 가능한 인공지능(XAI)"과 "감정 인식의 윤리적 기준"을 연구하며 인간 중심의 기술 철학을 확립하고 있습니다. 기술이 인간의 도덕을 대체할 수 없으며, 기술이 정의를 침범하지 않도록 지키는 것이 중요하다고 강조합니다. [출처](https://www.kairnews.

## 7. 대화 히스토리를 유지하는 Web 검색 Agent

`chat_history` 를 누적하면서, 이전 답변(웹 검색 결과 포함)을 기억하고 이어 답변하는지 확인합니다.

In [31]:
chat_history = []


def chat(user_input: str):
    answer = run_agent(
        user_input,
        tools=all_tools,
        system=SYSTEM,
        chat_history=chat_history,
        verbose=False,
    )
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=answer))
    print(f"사용자: {user_input}")
    print(f"Agent : {answer}")
    print("-" * 60)


# 연속 대화 — 도구가 자연스럽게 전환되는지 확인
chat("모델을 만든 구글 논문 'Attention Is All You Need' 관련 최신 글을 웹에서 찾아봐.")  # web_search
chat("내일 학회 발표가 있는 샌프란시스코의 현재 날씨도 알려줘.")        # get_weather

사용자: 모델을 만든 구글 논문 'Attention Is All You Need' 관련 최신 글을 웹에서 찾아봐.
Agent : 구글의 논문 'Attention Is All You Need'는 Transformer 모델을 제안한 기념비적인 연구로, 자연어 처리 분야에 큰 혁신을 가져왔습니다. 이 논문은 기존의 순환 신경망(RNN) 기반 모델들과 달리, 순차적 처리를 제거하고 전체 시퀀스를 한 번에 처리할 수 있는 구조를 도입하여 학습 효율성과 성능을 크게 향상시켰습니다. 최신 글에서는 이 논문의 핵심 개념인 Scaled Dot-Product Attention과 Multi-Head Attention에 대해 설명하고 있습니다. [자세한 내용 보기](https://aliencoder.tistory.com/216)
------------------------------------------------------------
사용자: 내일 학회 발표가 있는 샌프란시스코의 현재 날씨도 알려줘.
Agent : 현재 샌프란시스코의 날씨 정보를 가져오는 데 문제가 발생했습니다. 잠시 후 다시 시도해 주시거나 다른 방법으로 확인해 보시기 바랍니다.
------------------------------------------------------------


## 정리

이 노트북에서 사용한 도구는 모두 **API 키 발급이 불필요한 무료 공개 API**입니다. 날씨 정보를 가져오는데 오류가 발생할 수도 있습니다.

| 도구 | 외부 API | LangChain 래퍼 | 키 필요? |
|------|---------|----------------|---------|
| `web_search` | DuckDuckGo | `DuckDuckGoSearchResults` | ❌ |
| `get_weather` | Open-Meteo geocoding + forecast | `requests` 직접 호출 | ❌ |


**고도화 방법**
- 키가 필요한 API로 확장: OpenWeatherMap(상세 날씨), NewsAPI/Tavily(고품질 뉴스), Google CSE